# A transformer block from scratch

A building-inspection firm wants a model that tags inspectors' notes. The team inherits a half-written PyTorch
file that crashes inside the first transformer block. In this notebook you write the block, trace its shapes,
break it and fix it, and compare it with PyTorch's own `nn.TransformerEncoderLayer`. Same notes, same seed and
same code as the lesson.

About 10 minutes. CPU only, nothing to install.

## Step 1: the notes and the sizes

This cell imports PyTorch and writes down the two inspection notes, six words each. Every word becomes a vector
of 16 numbers. The block has 4 attention heads and a feed-forward layer 64 numbers wide.

**Look for:** the PyTorch version. The lesson was run with 2.15.0. Another version can change the last digits
of a small number later on, but not the shapes.

In [1]:
import inspect, sys, json
import torch
from torch import nn

torch.manual_seed(0)
print("torch", torch.__version__)

NOTES = [
    "rust on balcony rail near door".split(),
    "hairline crack above east window lintel".split(),
]
D_MODEL, HEADS, D_FF = 16, 4, 64
print("notes:", NOTES)

torch 2.15.0
notes: [['rust', 'on', 'balcony', 'rail', 'near', 'door'], ['hairline', 'crack', 'above', 'east', 'window', 'lintel']]


## Step 2: the block

This is the team's block once it runs. `__init__` makes the layers: one `Linear` for q, k and v together, one
`Linear` after the heads are merged, two `LayerNorm`s, and the feed-forward pair `up` (16 to 64) and `down`
(64 back to 16). `block_forward` is the whole block in 10 lines: attention, add, layer norm, feed-forward, add,
layer norm. There is no causal mask: a tagger reads the whole note at once.

**Look for:** nothing is printed yet. Read lines 7 and 9 of `block_forward`: each one adds a sublayer's output to
its input. That only works if the sublayer hands back 16 numbers per word.

In [2]:
class Block(nn.Module):
    def __init__(self, d_model=16, heads=4, d_ff=64):
        super().__init__()
        self.heads, self.dh = heads, d_model // heads
        self.qkv = nn.Linear(d_model, 3 * d_model)   # q, k and v in one layer
        self.proj = nn.Linear(d_model, d_model)      # after the heads are merged
        self.norm1 = nn.LayerNorm(d_model)
        self.up = nn.Linear(d_model, d_ff)           # feed-forward: widen ...
        self.down = nn.Linear(d_ff, d_model)         # ... and come back
        self.norm2 = nn.LayerNorm(d_model)

    def split_heads(self, z):   # (b, t, d) -> (b, heads, t, dh)
        b, t, d = z.shape
        return z.view(b, t, self.heads, self.dh).transpose(1, 2)

    def merge_heads(self, z):   # (b, heads, t, dh) -> (b, t, d)
        b, h, t, dh = z.shape
        return z.transpose(1, 2).reshape(b, t, h * dh)

    def forward(self, notes):
        return block_forward(self, notes)


def block_forward(blk, notes):
    qkv = blk.qkv(notes)
    q, k, v = qkv.chunk(3, dim=-1)
    q, k, v = map(blk.split_heads, (q, k, v))
    scores = q @ k.transpose(-2, -1) / blk.dh ** 0.5
    weights = scores.softmax(dim=-1)
    mixed = blk.merge_heads(weights @ v)
    notes = blk.norm1(notes + blk.proj(mixed))
    wide = blk.up(notes).relu()
    notes = blk.norm2(notes + blk.down(wide))
    return notes

## Step 3: turn the notes into numbers and run the block once

`make_input` gives each word a random 16-number vector (an `nn.Embedding`, seeded), so the input is
(notes, words, numbers per word). Then we make a block with seed 0 and run it.

**Look for:** input shape `(2, 6, 16)` and output shape `(2, 6, 16)`. The block hands back its input's shape.

In [3]:
def make_input(seed=0, d_model=16):
    torch.manual_seed(seed)
    vocab = sorted({w for n in NOTES for w in n})
    emb = nn.Embedding(len(vocab), d_model)
    ids = torch.tensor([[vocab.index(w) for w in n] for n in NOTES])
    with torch.no_grad():
        return emb(ids)


torch.manual_seed(0)
blk = Block().eval()
notes = make_input(seed=0)
with torch.no_grad():
    out = blk(notes)
print("input shape ", tuple(notes.shape))
print("output shape", tuple(out.shape))

input shape  (2, 6, 16)
output shape (2, 6, 16)


## Step 4: trace the block line by line

A small tracer runs `block_forward` and records the shape of every local tensor before each line runs. We print
the listing, then one row per step: the line that just ran and the tensors it made or changed.

**Look for:** `qkv` `(2, 6, 48)`; `q` first `(2, 6, 16)`, then `(2, 4, 6, 4)` after the split into heads;
`scores` `(2, 4, 6, 6)` (words by words, per note and head); `wide` `(2, 6, 64)`; and the return value
`(2, 6, 16)`, the input's shape. These are the shapes you predicted on the lesson's first two screens.

In [4]:
def shape_of(v):
    if isinstance(v, torch.Tensor):
        return list(v.shape)
    if isinstance(v, (int, float, str)):
        return v
    return None


def trace_run(fn, *args, keep=None):
    """Runs fn and records, before each line, the shapes of its local tensors."""
    steps = []
    first = fn.__code__.co_firstlineno
    def local(frame, event, arg):
        if event in ("line", "return"):
            vs = {k: shape_of(v) for k, v in frame.f_locals.items() if (keep is None or k in keep)}
            vs = {k: v for k, v in vs.items() if v is not None}
            steps.append({"line": frame.f_lineno - first, "event": event, "vars": vs})
        return local
    sys.settrace(lambda frame, event, arg: local if frame.f_code is fn.__code__ else None)
    try:
        result = fn(*args)
    finally:
        sys.settrace(None)
    return result, steps


listing = inspect.getsource(block_forward).rstrip().split("\n")
print("--- listing (line 0 = def) ---")
for i, ln in enumerate(listing):
    print(i, ln)

with torch.no_grad():
    out, steps = trace_run(block_forward, blk, notes)

print("--- trace: line that ran -> tensors it made or changed ---")
for prev, cur in zip(steps, steps[1:]):
    if cur["event"] == "return":
        print(f"line {prev['line']:2d} returns notes {tuple(cur['vars']['notes'])}")
        break
    new = {k: v for k, v in cur["vars"].items() if prev["vars"].get(k) != v}
    if prev["line"] in (7, 9):           # notes is replaced by a tensor of the same shape
        new = {"notes": cur["vars"]["notes"]}
    print(f"line {prev['line']:2d}", ", ".join(f"{k} {tuple(v)}" for k, v in new.items()))
print("output shape", tuple(out.shape))

--- listing (line 0 = def) ---
0 def block_forward(blk, notes):
1     qkv = blk.qkv(notes)
2     q, k, v = qkv.chunk(3, dim=-1)
3     q, k, v = map(blk.split_heads, (q, k, v))
4     scores = q @ k.transpose(-2, -1) / blk.dh ** 0.5
5     weights = scores.softmax(dim=-1)
6     mixed = blk.merge_heads(weights @ v)
7     notes = blk.norm1(notes + blk.proj(mixed))
8     wide = blk.up(notes).relu()
9     notes = blk.norm2(notes + blk.down(wide))
10     return notes
--- trace: line that ran -> tensors it made or changed ---
line  1 qkv (2, 6, 48)
line  2 q (2, 6, 16), k (2, 6, 16), v (2, 6, 16)
line  3 q (2, 4, 6, 4), k (2, 4, 6, 4), v (2, 4, 6, 4)
line  4 scores (2, 4, 6, 6)
line  5 weights (2, 4, 6, 6)
line  6 mixed (2, 6, 16)
line  7 notes (2, 6, 16)
line  8 wide (2, 6, 64)
line  9 notes (2, 6, 16)
line 10 returns notes (2, 6, 16)
output shape (2, 6, 16)


## Step 5: change one word and see who notices

We give one word, "crack" in note 2, a different vector and keep everything else. Then we run each sublayer on
the original notes and on the changed notes, and mark which output rows changed (1 = changed). Each row is one
word.

**Look for:** after the attention sublayer, all `6` rows of note 2 change and `0` rows of note 1. After the
feed-forward sublayer alone, only `1` row of note 2 changes: the word that changed. Attention mixes the words of
one note; the feed-forward layer works on each word alone; the notes never mix.

In [5]:
changed = notes.clone()
torch.manual_seed(1)
changed[1, 1] = torch.randn(16)      # note 2, word 2 ('crack')


def attn_sublayer(x):
    q, k, v = map(blk.split_heads, blk.qkv(x).chunk(3, dim=-1))
    w = (q @ k.transpose(-2, -1) / blk.dh ** 0.5).softmax(dim=-1)
    return blk.norm1(x + blk.proj(blk.merge_heads(w @ v)))


def ffn_sublayer(x):
    return blk.norm2(x + blk.down(blk.up(x).relu()))


with torch.no_grad():
    for name, f in [("attention sublayer", attn_sublayer), ("feed-forward sublayer", ffn_sublayer)]:
        a, b = f(notes), f(changed)
        diff = (a - b).abs().amax(dim=-1)  # (notes, words)
        rows = (diff > 1e-6).int().tolist()
        print(name, "rows changed per note (1 = changed):", rows, "count per note:", [sum(r) for r in rows])

attention sublayer rows changed per note (1 = changed): [[0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1]] count per note: [0, 6]
feed-forward sublayer rows changed per note (1 = changed): [[0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0]] count per note: [0, 1]


## Step 6: where the parameters are

We count the numbers each layer learns.

**Look for:** a total of `3280`. `up` (1088) and `down` (1040) together hold about 65% of them: the feed-forward
layer is where most weights live.

In [6]:
parts = {"qkv": blk.qkv, "proj": blk.proj, "norm1": blk.norm1, "up": blk.up, "down": blk.down, "norm2": blk.norm2}
for n, m in parts.items():
    print(f"{n:6s} {sum(p.numel() for p in m.parameters())}")
total = sum(p.numel() for p in blk.parameters())
ffn = sum(p.numel() for p in blk.up.parameters()) + sum(p.numel() for p in blk.down.parameters())
print("total ", total)
print(f"feed-forward share: {ffn / total:.0%}")

qkv    816
proj   272
norm1  32
up     1088
down   1040
norm2  32
total  3280
feed-forward share: 65%


## Step 7: the inherited file, broken on purpose

This is the half-written file's block: `down` is `Linear(64, 64)` instead of `Linear(64, 16)`. Forward hooks print
each layer's output shape as it runs. The crash is caught and printed, so the notebook keeps going.

**Look for:** `up` gives `(2, 6, 64)` as planned, but `down` also gives `(2, 6, 64)`. It should come back to 16.
The error then appears one step later, at the add on line 9: `RuntimeError: The size of tensor a (16) must match
the size of tensor b (64) at non-singleton dimension 2`. The add is the symptom; `down` is the cause.

In [7]:
import traceback


class BrokenBlock(Block):
    def __init__(self, d_model=16, heads=4, d_ff=64):
        super().__init__(d_model, heads, d_ff)
        self.down = nn.Linear(d_ff, d_ff)   # the inherited file says Linear(64, 64)


torch.manual_seed(0)
broken = BrokenBlock().eval()
for name in ["qkv", "proj", "norm1", "up", "down", "norm2"]:
    getattr(broken, name).register_forward_hook(
        lambda m, i, o, name=name: print(f"{name:6s} {m}  ->  output {tuple(o.shape)}"))
print("input  notes  ->  ", tuple(notes.shape))
try:
    with torch.no_grad():
        broken(notes)
except Exception as e:
    last = traceback.extract_tb(e.__traceback__)[-1]
    print(f"in {last.name}, line: {last.line}")
    print(f"{type(e).__name__}: {e}")

input  notes  ->   (2, 6, 16)
qkv    Linear(in_features=16, out_features=48, bias=True)  ->  output (2, 6, 48)
proj   Linear(in_features=16, out_features=16, bias=True)  ->  output (2, 6, 16)
norm1  LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)  ->  output (2, 6, 16)
up     Linear(in_features=16, out_features=64, bias=True)  ->  output (2, 6, 64)
down   Linear(in_features=64, out_features=64, bias=True)  ->  output (2, 6, 64)
in block_forward, line: notes = blk.norm2(notes + blk.down(wide))
RuntimeError: The size of tensor a (16) must match the size of tensor b (64) at non-singleton dimension 2


## Step 8: fix it

The fix is one line in `__init__`: `self.down = nn.Linear(d_ff, d_model)`, so the feed-forward layer brings each
word back to 16 numbers. That is exactly our `Block`. We make the fixed block with the same seed and the same
hooks.

**Look for:** `down` now gives `(2, 6, 16)`, `norm2` runs, and the block returns `(2, 6, 16)`.

In [8]:
class FixedBlock(BrokenBlock):
    def __init__(self, d_model=16, heads=4, d_ff=64):
        super().__init__(d_model, heads, d_ff)
        self.down = nn.Linear(d_ff, d_model)   # the fix: back to the model's width


torch.manual_seed(0)
fixed = FixedBlock().eval()
for name in ["up", "down", "norm2"]:
    getattr(fixed, name).register_forward_hook(
        lambda m, i, o, name=name: print(f"{name:6s} {m}  ->  output {tuple(o.shape)}"))
with torch.no_grad():
    print("fixed block returns", tuple(fixed(notes).shape))

up     Linear(in_features=16, out_features=64, bias=True)  ->  output (2, 6, 64)
down   Linear(in_features=64, out_features=16, bias=True)  ->  output (2, 6, 16)
norm2  LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)  ->  output (2, 6, 16)
fixed block returns (2, 6, 16)


## Step 9: the same block from the library

PyTorch ships this block as `nn.TransformerEncoderLayer`. We make one with the same sizes, `dropout=0.0` and
`batch_first=True`, copy our block's weights into it, and run both on the same notes.

**Look for:** `norm_first: False` (layer norm after each add, like ours); `3280` parameters in both; output
`(2, 6, 16)`; and a largest difference around `2.4e-07`. That is float rounding, not different math: the two are
the same block.

In [9]:
torch.manual_seed(0)
ours = Block().eval()
notes = make_input(seed=0)


def make_layer(batch_first, norm_first=False):
    layer = nn.TransformerEncoderLayer(d_model=16, nhead=4, dim_feedforward=64, dropout=0.0,
                                       batch_first=batch_first, norm_first=norm_first)
    with torch.no_grad():
        layer.self_attn.in_proj_weight.copy_(ours.qkv.weight)
        layer.self_attn.in_proj_bias.copy_(ours.qkv.bias)
        layer.self_attn.out_proj.weight.copy_(ours.proj.weight)
        layer.self_attn.out_proj.bias.copy_(ours.proj.bias)
        layer.linear1.load_state_dict(ours.up.state_dict())
        layer.linear2.load_state_dict(ours.down.state_dict())
        layer.norm1.load_state_dict(ours.norm1.state_dict())
        layer.norm2.load_state_dict(ours.norm2.state_dict())
    return layer.eval()


layer = make_layer(batch_first=True)
print(layer)
print("norm_first:", layer.norm_first)
print("parameters, library layer:", sum(p.numel() for p in layer.parameters()))
print("parameters, our block:   ", sum(p.numel() for p in ours.parameters()))
with torch.no_grad():
    a = ours(notes)
    b = layer(notes)
print("input shape", tuple(notes.shape), "-> library output shape", tuple(b.shape))
print("largest difference, ours vs library (batch_first=True):", f"{(a - b).abs().max().item():.1e}")

TransformerEncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
  )
  (linear1): Linear(in_features=16, out_features=64, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (linear2): Linear(in_features=64, out_features=16, bias=True)
  (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)
  (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)
  (dropout1): Dropout(p=0.0, inplace=False)
  (dropout2): Dropout(p=0.0, inplace=False)
)
norm_first: False
parameters, library layer: 3280
parameters, our block:    3280
input shape (2, 6, 16) -> library output shape (2, 6, 16)
largest difference, ours vs library (batch_first=True): 2.4e-07


## Step 10: the quiet bug

A teammate leaves out `batch_first=True`. The default is `False`, so the layer reads the input as
(words, notes, numbers): it takes our 2 notes as 2 "words" and our 6 words as 6 separate examples. Nothing
fails. Then we run the test from the lesson: change one word of note 2 and count how many of note 1's rows move.

**Look for:** output shape `(2, 6, 16)`, no error, but a largest difference of about `0.68` from the right
answer. In the test, note 1 moves `0` rows with `batch_first=True` and `1` row with `batch_first=False`: word 2 of
note 1 now attends to word 2 of note 2. The notes mixed.

In [10]:
layer_f = make_layer(batch_first=False)
print("batch_first:", layer_f.self_attn.batch_first)
with torch.no_grad():
    c = layer_f(notes)
print("output shape", tuple(c.shape))
print("largest difference, ours vs library (batch_first=False):", f"{(a - c).abs().max().item():.2f}")

print("--- test: change 'crack' in note 2; does note 1 move? ---")
with torch.no_grad():
    for name, lyr in [("batch_first=True ", layer), ("batch_first=False", layer_f)]:
        moved = (lyr(notes) - lyr(changed)).abs().amax(dim=-1)[0] > 1e-6
        print(name, "note 1 rows that moved:", moved.int().tolist(), "count:", int(moved.sum()))

batch_first: False
output shape (2, 6, 16)
largest difference, ours vs library (batch_first=False): 0.68
--- test: change 'crack' in note 2; does note 1 move? ---
batch_first=True  note 1 rows that moved: [0, 0, 0, 0, 0, 0] count: 0
batch_first=False note 1 rows that moved: [0, 1, 0, 0, 0, 0] count: 1


## Now change it

Three small experiments. In each cell, edit the line marked `# <-- change this`, run the cell, and compare.

### Experiment 1: long notes

Inspectors start pasting whole reports. Set `WORDS` to 600 (it is 6 now). The cell feeds random notes of that
length through the block and counts the numbers in three tensors.

**What should change:** `qkv` and `wide` grow 100 times (600 / 6), because they have one row per word. `scores`
grows 10,000 times, because it has the words twice: words by words. That is why long inputs are the expensive
case for attention. The parameter count does not change at all.

In [11]:
WORDS = 6   # <-- change this: try 600

torch.manual_seed(0)
long_notes = torch.randn(2, WORDS, D_MODEL)
with torch.no_grad():
    _, st = trace_run(block_forward, blk, long_notes, keep=["qkv", "wide", "scores"])
final = st[-1]["vars"]
for name in ["qkv", "wide", "scores"]:
    s = final[name]
    n = 1
    for x in s:
        n *= x
    print(f"{name:6s} shape {tuple(s)}  numbers {n:,}")
print("parameters", sum(p.numel() for p in blk.parameters()))

qkv    shape (2, 6, 48)  numbers 576
wide   shape (2, 6, 64)  numbers 768
scores shape (2, 4, 6, 6)  numbers 288
parameters 3280


### Experiment 2: layer norm before the add

`norm_first=True` moves each layer norm in front of its sublayer: `x + sublayer(norm(x))` instead of
`norm(x + sublayer(x))`. Set `NORM_FIRST` to `True` and compare the library layer with our post-norm block again,
with the same copied weights.

**What should change:** the largest difference jumps from about `2.4e-07` to a large number. The weights are the
same, but the order of the steps is not, so it is a different block. The shape stays `(2, 6, 16)`.

In [12]:
NORM_FIRST = False   # <-- change this: try True

layer_n = make_layer(batch_first=True, norm_first=NORM_FIRST)
with torch.no_grad():
    d = layer_n(notes)
print("norm_first:", layer_n.norm_first, " output shape", tuple(d.shape))
print("largest difference, ours vs library:", f"{(a - d).abs().max().item():.1e}")

norm_first: False  output shape (2, 6, 16)
largest difference, ours vs library: 2.4e-07


### Experiment 3: a different number of heads

Set `H` to 2 or 8 (it is 4 now). Each head gets `16 / H` numbers per word.

**What should change:** `q` after the split becomes `(2, H, 6, 16 / H)` and `scores` becomes `(2, H, 6, 6)`: one
words-by-words grid per head. The output is still `(2, 6, 16)`, and the parameter count stays `3280`, because
the heads only cut the same 16 numbers into different slices.

In [13]:
H = 4   # <-- change this: try 2 or 8

torch.manual_seed(0)
blk_h = Block(heads=H).eval()
with torch.no_grad():
    out_h, st = trace_run(block_forward, blk_h, notes, keep=["q", "scores"])
final = st[-1]["vars"]
print("q after split", tuple(final["q"]))
print("scores       ", tuple(final["scores"]))
print("output       ", tuple(out_h.shape))
print("parameters   ", sum(p.numel() for p in blk_h.parameters()))

q after split (2, 4, 6, 4)
scores        (2, 4, 6, 6)
output        (2, 6, 16)
parameters    3280


## What you should see

- The traced block prints `scores` as `(2, 4, 6, 6)`, `wide` as `(2, 6, 64)`, and returns `(2, 6, 16)`.
- Changing "crack" changes 6 of note 2's rows after attention, 1 after the feed-forward layer, and 0 of note 1's.
- The broken block (`down = Linear(64, 64)`) raises `The size of tensor a (16) must match the size of tensor b (64)
  at non-singleton dimension 2`; with `Linear(64, 16)` it returns `(2, 6, 16)`.
- Your block and `nn.TransformerEncoderLayer` both count 3,280 parameters, and differ by about 2.4e-07 with
  `batch_first=True`, but by 0.68 without it.